In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision.transforms import v2
from torch.utils.data.dataloader import DataLoader
from torch.profiler import profile, ProfilerActivity

In [2]:
transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

batch_size = 2**1

trainset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=0)


testset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=False, num_workers=0)

In [3]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"

In [4]:
# import matplotlib.pyplot as plt
# import numpy as np

# def imshow(img):
#     img = img / 2 + 0.5 #unnormalize
#     npimg = img.numpy()
#     plt.imshow(np.transpose(npimg, (1,2,0)))
#     plt.show()
# #get random training images    
# dataiter = iter(trainloader)
# images, labels = next(dataiter)

# imshow(torchvision.utils.make_grid(images))

In [5]:
# print(labels)
# print(images.shape)

In [6]:
def conv2d_out(size, kernel_size, stride=1, padding=0, dilation=1):
    return (size + 2 * padding - dilation * (kernel_size - 1) - 1) // stride + 1

def pool2d_out(size, kernel_size=2, stride=2, padding=0, dilation=1):
    return (size + 2 * padding - dilation * (kernel_size - 1) - 1) // stride + 1

In [7]:
class Net(nn.Module):
    def __init__(self, im_shape: list[int] = [3, 28, 28], kernel_sizes: int = 5, out_channels: list[int] = [8, 16], linear_layers: list[int] = [128, 64, 10]):
        super().__init__()
        self.conv1 = nn.Conv2d(im_shape[0], out_channels=out_channels[0], kernel_size=kernel_sizes)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(in_channels=out_channels[0], out_channels=out_channels[1], kernel_size=kernel_sizes)
        
        self.fc1 = nn.LazyLinear(out_features=linear_layers[0])
        self.fc2 = nn.Linear(in_features=linear_layers[0], out_features=linear_layers[1])
        self.fc3 = nn.Linear(in_features=linear_layers[1], out_features=linear_layers[2])

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1) # flatten all dimensions except batch
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [160]:
f.shape

torch.Size([10, 2, 12, 12])

In [139]:
d_f = 12
kernel_size = 5
padding = (kernel_size-1) // 2

In [140]:
cnn = nn.Conv2d(in_channels=2, out_channels=2, kernel_size=kernel_size, dilation=1, padding=padding)

In [141]:
cnn._parameters['weight'].shape

torch.Size([2, 2, 5, 5])

In [142]:
for param in cnn.parameters():
    print(param)

Parameter containing:
tensor([[[[-0.0270,  0.1221,  0.0698, -0.0235,  0.0598],
          [-0.0763,  0.0889,  0.0340, -0.0747, -0.1373],
          [-0.1078, -0.0233, -0.0705,  0.1060, -0.0380],
          [ 0.0885, -0.0744,  0.0203, -0.0257,  0.1314],
          [-0.0481, -0.1364, -0.0168, -0.0846, -0.0648]],

         [[ 0.0174,  0.0663, -0.0045, -0.0526,  0.1025],
          [ 0.0330, -0.0287, -0.1032, -0.0863,  0.0804],
          [-0.0023, -0.0014,  0.1286, -0.0231, -0.1100],
          [-0.0435, -0.0769,  0.1384,  0.0804,  0.0541],
          [ 0.1158,  0.0672, -0.0800, -0.0128,  0.0837]]],


        [[[-0.1136,  0.1338,  0.1022,  0.0879, -0.1101],
          [ 0.0301,  0.1216,  0.1275, -0.1093,  0.0394],
          [ 0.0654, -0.0284, -0.0008, -0.1384,  0.1389],
          [ 0.1092,  0.0545, -0.1192, -0.0578, -0.0597],
          [-0.0356, -0.0106,  0.0900,  0.0026, -0.0795]],

         [[-0.0303,  0.0109, -0.1186,  0.0510,  0.0059],
          [ 0.1062, -0.0347, -0.0072,  0.1390, -0.1179],
 

In [143]:
f = torch.zeros(size=(10, 2, d_f, d_f))

In [144]:
out = cnn(f)

In [158]:
o_shape, f_shape = out.shape[-2:], f.shape[-2:]

print(f_shape[0] - ((kernel_size - 1) + (2 * padding)) == o_shape[0], f_shape[1] - ((kernel_size - 1) + (2 * padding)) == o_shape[1])

False False


In [87]:
loss_fn = torch.nn.CrossEntropyLoss()
model: nn.Module = Net().to(device=device)
optimizer = torch.optim.SGD(params=model.parameters(), lr=0.001, momentum=0.9)

In [88]:
profiled_once = False

for epoch in range(2):
    running_loss = 0.0

    for i, data in enumerate(trainloader, 0):
        inputs, labels = data
        inputs, labels = inputs.to(device=device), labels.to(device=device)

        optimizer.zero_grad()

        if not profiled_once:
            with profile(
                activities=[ProfilerActivity.CPU],
                with_flops=True
            ) as prof:
                outputs = model(inputs)
                loss = loss_fn(outputs, labels)
                loss.backward()

            batch_flops = sum(evt.flops for evt in prof.key_averages() if evt.flops is not None)

            print(f"Approx FLOPs / batch: {batch_flops:,}")
            print(f"Approx FLOPs / image: {batch_flops / inputs.size(0):,.0f}")

            profiled_once = True
        else:
            outputs = model(inputs)
            loss = loss_fn(outputs, labels)
            loss.backward()

        optimizer.step()

        running_loss += loss.item()

Approx FLOPs / batch: 2,201,600
Approx FLOPs / image: 1,100,800


In [89]:
correct = 0
total = 0

model.eval()
with torch.no_grad():
    for inputs, labels in testloader:
        inputs, labels = inputs.to(device=device), labels.to(device=device)
        outputs = model(inputs)
        preds = outputs.argmax(dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total
print(f"Accuracy: {accuracy:.3f}")

Accuracy: 0.988
